# 第 6 周练习：“价格是正确的”——改进微调
## 作者：瓦坎达团队的 Mougang Thomas Gasmyr

该笔记本通过微调 **GPT-4.1-nano** 来实现改进的产品价格预测模型：

- **平衡价格抽样** — 不同价格范围内的代表性相同，以避免对廉价商品的偏见
- **专家系统提示** — 领域感知定价分析师角色
- **更大的训练集** — 3,000 个平衡示例（5 美元预算内）
- **评估和比较** - 针对基线和前沿模型
- **HuggingFace Hub** — 将精选数据集推送到个人存储库

### 策略
原始笔记本使用随机抽样，偏向低价商品（分布严重倾斜）。
通过平衡价格区间并使用领域专家提示，我们帮助模型了解完整的价格范围。

## 设置

In [ ]:
import os
import re
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

import sys
sys.path.insert(0, '../../week6')
from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

openai = OpenAI()

%matplotlib inline

## 配置

对于较小的数据集（22K 项）设置“LITE_MODE = True”，对于完整数据集（820K 项）设置“False”。

In [ ]:
# 配置
# Configuration

LITE_MODE = False
HF_USERNAME = "Gasmyr"

TRAINING_SIZE = 3000
VALIDATION_SIZE = 300
N_EPOCHS = 1
BASE_MODEL = "gpt-4.1-nano-2025-04-14"

print(f"Mode: {'Lite' if LITE_MODE else 'Full'}")
print(f"Training size: {TRAINING_SIZE:,}")
print(f"Validation size: {VALIDATION_SIZE:,}")
print(f"Base model: {BASE_MODEL}")
print(f"Epochs: {N_EPOCHS}")

## 第 1 步：加载数据

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# Inspect a sample item

print(f"Title: {train[0].title}")
print(f"Price: ${train[0].price:.2f}")
print(f"Category: {train[0].category}")
print(f"\nSummary:\n{train[0].summary}")

## 步骤 2：分析价格分布

了解价格偏差是我们改进策略的关键。

In [ ]:
train_prices = [item.price for item in train]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw distribution
axes[0].hist(train_prices, bins=range(0, 1000, 10), color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title(f'Training Price Distribution (n={len(train):,})')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(train_prices), color='red', linestyle='--', label=f'Mean: ${np.mean(train_prices):.0f}')
axes[0].axvline(np.median(train_prices), color='green', linestyle='--', label=f'Median: ${np.median(train_prices):.0f}')
axes[0].legend()

# Price buckets
def categorize_price(price):
    if price < 50:
        return '$0-50'
    elif price < 100:
        return '$50-100'
    elif price < 200:
        return '$100-200'
    elif price < 400:
        return '$200-400'
    else:
        return '$400+'

bucket_counts = Counter([categorize_price(p) for p in train_prices])
bucket_order = ['$0-50', '$50-100', '$100-200', '$200-400', '$400+']
counts = [bucket_counts[b] for b in bucket_order]

axes[1].bar(bucket_order, counts, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('Items per Price Bucket (Imbalanced!)')
axes[1].set_xlabel('Price Range')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nThe distribution is heavily skewed: {bucket_counts['$0-50']:,} items under $50 vs {bucket_counts['$400+']:,} over $400")
print("Random sampling would bias the model toward cheap items. Balanced sampling fixes this.")

## 步骤 3：创建平衡的训练和验证集

我们不是随机抽样，而是从每个价格桶中均匀抽样，以便模型学习完整的价格范围。

In [ ]:
def create_balanced_dataset(items, size, seed=42):
    """Sample equally from each price bucket."""
    random.seed(seed)

    buckets = defaultdict(list)
    for item in items:
        buckets[categorize_price(item.price)].append(item)

    per_bucket = size // len(buckets)
    selected = []

    for bucket_name in bucket_order:
        bucket_items = buckets[bucket_name]
        sample_size = min(per_bucket, len(bucket_items))
        selected.extend(random.sample(bucket_items, sample_size))

    random.shuffle(selected)
    return selected


fine_tune_train = create_balanced_dataset(train, TRAINING_SIZE, seed=42)
fine_tune_val = create_balanced_dataset(val, VALIDATION_SIZE, seed=43)

print(f"Training set: {len(fine_tune_train):,} items")
print(f"Validation set: {len(fine_tune_val):,} items")

# Verify balance
for name, dataset_items in [("Training", fine_tune_train), ("Validation", fine_tune_val)]:
    bc = Counter([categorize_price(item.price) for item in dataset_items])
    print(f"\n{name} balance:")
    for b in bucket_order:
        print(f"  {b}: {bc[b]} ({bc[b]/len(dataset_items)*100:.0f}%)")

In [ ]:
# Visualize the balanced distribution

balanced_prices = [item.price for item in fine_tune_train]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(train_prices[:TRAINING_SIZE], bins=range(0, 1000, 20), color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title(f'Random Sampling ({TRAINING_SIZE:,} items)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(balanced_prices, bins=range(0, 1000, 20), color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].set_title(f'Balanced Sampling ({len(fine_tune_train):,} items)')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')

plt.suptitle('Random vs Balanced Sampling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 步骤 4：设计专家系统提示

我们使用领域专家角色为模型提供更好的价格估计背景。

In [ ]:
SYSTEM_PROMPT = (
    "You are a pricing analyst with expertise in consumer electronics, appliances, automotive parts, "
    "musical instruments, and retail products. Based on the product's features, brand, specifications, "
    "and category, estimate the typical retail price. Respond only with the price, e.g. $123.45"
)

print("System prompt:")
print(SYSTEM_PROMPT)

In [ ]:
def messages_for(item):
    """Create training messages: user asks for price, assistant responds with actual price."""
    user_content = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]


def test_messages_for(item):
    """Create inference messages: no assistant response (model predicts)."""
    user_content = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


# Preview
print("Example training message:")
print(json.dumps(messages_for(fine_tune_train[0]), indent=2))

## 步骤 5：准备 JSONL 文件并上传到 OpenAI

In [ ]:
def make_jsonl(items):
    lines = []
    for item in items:
        msgs = messages_for(item)
        lines.append(json.dumps({"messages": msgs}))
    return "\n".join(lines)


def write_jsonl(items, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(items))
    print(f"Written {len(items):,} items to {filename}")


write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_val, "fine_tune_validation.jsonl")

In [ ]:
# Upload files to OpenAI

with open("fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
print(f"Training file uploaded: {train_file.id}")

with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
print(f"Validation file uploaded: {validation_file.id}")

## 第 6 步：启动微调作业

微调 GPT-4.1-nano 非常经济高效（约 0.10 美元/100 万训练代币）。
有 3,000 个示例，每个示例约 300 个代币，成本应该远低于 1 美元。

In [ ]:
fine_tune_job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=BASE_MODEL,
    seed=42,
    hyperparameters={"n_epochs": N_EPOCHS, "batch_size": "auto"},
    suffix="pricer-balanced"
)

job_id = fine_tune_job.id
print(f"Fine-tuning job created: {job_id}")
print(f"Status: {fine_tune_job.status}")

In [ ]:
# Monitor the job -- re-run this cell to check progress

status = openai.fine_tuning.jobs.retrieve(job_id)
print(f"Status: {status.status}")

if status.fine_tuned_model:
    print(f"Model ready: {status.fine_tuned_model}")

if status.trained_tokens:
    print(f"Trained tokens: {status.trained_tokens:,}")

events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)
print("\nRecent events:")
for event in reversed(events.data):
    print(f"  {event.message}")

## 步骤 7：评估微调模型

In [ ]:
# Get the fine-tuned model name

fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

if fine_tuned_model_name:
    print(f"Fine-tuned model: {fine_tuned_model_name}")
else:
    print("Model not ready yet -- re-run the monitoring cell above and wait.")

In [ ]:
def get_price(s):
    """Extract a numeric price from model response."""
    s = s.replace('$', '').replace(',', '')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0


def gpt_4__1_nano_balanced(item):
    """Our improved fine-tuned model."""
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7,
        temperature=0,
        seed=42
    )
    return get_price(response.choices[0].message.content)


# Quick sanity check
sample = test[0]
pred = gpt_4__1_nano_balanced(sample)
print(f"Product: {sample.title[:60]}...")
print(f"Predicted: ${pred:.2f}")
print(f"Actual:    ${sample.price:.2f}")
print(f"Error:     ${abs(pred - sample.price):.2f}")

In [ ]:
# Full evaluation on 250 test items

print("Evaluating improved fine-tuned model (balanced + expert prompt)...\n")
evaluate(gpt_4__1_nano_balanced, test)

## 步骤 8：与基线比较（原始微调模型）

为了进行比较，我们也用原来的随机采样+基本提示的方式进行微调。

In [ ]:
# Baseline: random sampling with simple prompt (same size for fair comparison)


def baseline_messages_for(item):
    user_content = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]


# Random sample (first N items -- same as original notebook)
baseline_train = train[:TRAINING_SIZE]
baseline_val = val[:VALIDATION_SIZE]


def make_baseline_jsonl(items):
    lines = []
    for item in items:
        msgs = baseline_messages_for(item)
        lines.append(json.dumps({"messages": msgs}))
    return "\n".join(lines)


with open("baseline_train.jsonl", "w") as f:
    f.write(make_baseline_jsonl(baseline_train))
with open("baseline_val.jsonl", "w") as f:
    f.write(make_baseline_jsonl(baseline_val))

with open("baseline_train.jsonl", "rb") as f:
    baseline_train_file = openai.files.create(file=f, purpose="fine-tune")
with open("baseline_val.jsonl", "rb") as f:
    baseline_val_file = openai.files.create(file=f, purpose="fine-tune")

baseline_job = openai.fine_tuning.jobs.create(
    training_file=baseline_train_file.id,
    validation_file=baseline_val_file.id,
    model=BASE_MODEL,
    seed=42,
    hyperparameters={"n_epochs": N_EPOCHS, "batch_size": "auto"},
    suffix="pricer-baseline"
)

baseline_job_id = baseline_job.id
print(f"Baseline job created: {baseline_job_id}")
print(f"Status: {baseline_job.status}")

In [ ]:
# Monitor baseline job -- re-run to check progress

baseline_status = openai.fine_tuning.jobs.retrieve(baseline_job_id)
print(f"Status: {baseline_status.status}")
if baseline_status.fine_tuned_model:
    print(f"Model ready: {baseline_status.fine_tuned_model}")

In [ ]:
baseline_model_name = openai.fine_tuning.jobs.retrieve(baseline_job_id).fine_tuned_model


def gpt_4__1_nano_baseline(item):
    """Baseline fine-tuned model (random sampling, basic prompt)."""
    user_content = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    response = openai.chat.completions.create(
        model=baseline_model_name,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=7,
        temperature=0,
        seed=42
    )
    return get_price(response.choices[0].message.content)


print("Evaluating baseline fine-tuned model (random sampling, basic prompt)...\n")
evaluate(gpt_4__1_nano_baseline, test)

## 步骤 9：测试前沿模型（零样本比较）

我们还可以与没有任何微调的 GPT-4.1-nano 进行比较，看看微调有多大帮助。

In [ ]:
from litellm import completion as litellm_completion


def gpt_4__1_nano_zero_shot(item):
    """GPT-4.1-nano without fine-tuning (zero-shot)."""
    user_content = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    response = litellm_completion(
        model="openai/gpt-4.1-nano",
        messages=[{"role": "user", "content": user_content}]
    )
    return response.choices[0].message.content


print("Evaluating GPT-4.1-nano zero-shot (no fine-tuning)...\n")
evaluate(gpt_4__1_nano_zero_shot, test)

## 步骤 10：结果总结

In [ ]:
import plotly.graph_objects as go

results = [
    ("GPT-4.1-nano\nZero-Shot", "slateblue", 62.51),
    ("Baseline Fine-Tuned\n(Random Sampling)", "orange", 57.38),
    ("Improved Fine-Tuned\n(Balanced + Expert)", "green", 54.31),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="Week 6 Exercise: Model Comparison (Lower is Better)",
    yaxis=dict(range=[0, max(values) * 1.3], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=0),
    width=800,
    height=500
)
fig.show()

## 步骤 11：将精选数据推送到 HuggingFace Hub

将平衡微调数据集和完整数据集推送到您的个人 HuggingFace 存储库。

In [ ]:
# Push the balanced fine-tuning dataset

ft_dataset_name = f"{HF_USERNAME}/pricer-finetune-balanced"

Item.push_to_hub(
    ft_dataset_name,
    fine_tune_train,
    fine_tune_val,
    test[:1000]
)

print(f"Balanced fine-tuning dataset pushed to: https://huggingface.co/datasets/{ft_dataset_name}")

In [ ]:
# Push the full dataset to your repo

full_dataset_name = f"{HF_USERNAME}/items_full" if not LITE_MODE else f"{HF_USERNAME}/items_lite"

Item.push_to_hub(
    full_dataset_name,
    train,
    val,
    test
)

print(f"Full dataset pushed to: https://huggingface.co/datasets/{full_dataset_name}")

## 第 12 步：验证——从您的存储库加载

In [ ]:
# Verify we can load back from HuggingFace

my_train, my_val, my_test = Item.from_hub(ft_dataset_name)

print(f"Loaded from {ft_dataset_name}:")
print(f"  Train: {len(my_train):,} items")
print(f"  Val:   {len(my_val):,} items")
print(f"  Test:  {len(my_test):,} items")
print(f"\nSample: {my_train[0]}")

## 清理

In [ ]:
# Optional: clean up local JSONL files

for f in ["fine_tune_train.jsonl", "fine_tune_validation.jsonl", "baseline_train.jsonl", "baseline_val.jsonl"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 概括

### 我们做了什么
1. **分析**价格分布并发现廉价商品的严重偏向
2. **平衡** 5 个价格区间的训练集，以实现平等代表性
3. **设计**具有领域知识的专家系统提示
4. **微调** GPT-4.1-nano，具有 3,000 个平衡示例（1 epoch）
5. **与基线（随机采样）和零样本模型进行比较**
6. **将**所有数据集推送到 `Gasmyr/` 下的 HuggingFace Hub

### 关键见解
- 当目标变量倾斜时，**平衡采样**至关重要 - 它可以防止模型默认为众数
- **专家提示**提供领域上下文，即使是小模型也能做出更好的估计
- **GPT-4.1-nano** 对于微调来说极具成本效益（~0.10 美元/100 万代币）
- 即使是小模型的微调也可以在特定领域的任务上匹配或击败更大的零样本模型

### 成本
- 在 GPT-4.1-nano 上微调 3,000 个示例：~$0.10-$0.30
- 基线比较：~$0.10-$0.30
- 评估推断：~$0.05
- **总计：远低于 1 美元**（留下实验预算！）